# Retail Shelf Monitoring System — Model Training & Export Pipeline

This notebook trains and exports the two deep-learning models for the Retail Shelf Monitoring system:
1. **YOLO11 Product Detector** (Pre-trained on SKU-110K dense retail shelves + fine-tuned on custom store images)
2. **MobileNetV3-Large SKU Embedding Model & FAISS Index** (Metric learning on product crops)

### Instructions:
- **Runtime**: Select **GPU** (Colab: `Runtime` -> `Change runtime type` -> `T4 GPU` or Kaggle: `Accelerator` -> `GPU P100/T4`).
- **Expected Runtime**: ~45-90 minutes depending on chosen epochs and dataset fraction.
- **Input**: Upload `project_for_colab.zip` (generated locally with `python scripts/package_for_colab.py`).
- **Output**: Automatically downloads `shelf_models.zip`. Unzip it into your local `models/` directory.


In [ ]:
# =============================================================================
# CELL 1: TRAINING CONFIGURATION
# =============================================================================
SMOKE_TEST = False               # Set to True for quick 1-epoch dry run on CPU
EPOCHS_STAGE1 = 30              # Stage 1 pretraining on SKU-110K
EPOCHS_STAGE2 = 15              # Stage 2 fine-tuning on custom store data
EMBED_EPOCHS = 10               # Metric learning epochs for SKU crops
IMGSZ = 640                     # Input image resolution (640 recommended)
BATCH = 'auto'                  # Batch size ('auto' or integer e.g. 16)
MODEL_SIZE = 'yolo11s.pt'       # Base model: yolo11n.pt, yolo11s.pt, yolo11m.pt
SKU110K_FRACTION = 1.0          # Fraction of SKU-110K to use (e.g. 0.3 for faster pretraining)
RUN_ABLATION = True             # Run ablation: COCO vs SKU-110K vs Fine-tuned
SEED = 42

# Custom Dataset Configuration (Roboflow or uploaded zip)
# Leave blank to skip custom fine-tuning and use SKU-110K weights only
CUSTOM_DATA = {
    'zip_path': None,           # e.g. 'custom_store_data.zip'
    'roboflow_api_key': None,   # Roboflow API key (optional)
    'workspace': None,
    'project': None,
    'version': 1,
}

# Reference SKU Crops (class-per-folder)
SKU_CROPS_DIR = None            # e.g. 'sku_crops' folder or zip with class folders


In [ ]:
# =============================================================================
# CELL 2: ENVIRONMENT DETECTION & DEPENDENCIES
# =============================================================================
import os, sys
import torch

is_colab = 'google.colab' in sys.modules
is_kaggle = os.path.exists('/kaggle/working')
env_name = 'Google Colab' if is_colab else ('Kaggle' if is_kaggle else 'Standard Python')
print(f"Detected environment: {env_name}")

has_gpu = torch.cuda.is_available()
if has_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✓ GPU active: {gpu_name}")
else:
    if not SMOKE_TEST:
        raise SystemError("No GPU detected! Please enable GPU accelerator in Runtime settings or set SMOKE_TEST = True.")
    print("⚠ Running on CPU (SMOKE_TEST mode)")

# Install training dependencies
!pip install -q ultralytics onnx onnxruntime faiss-cpu roboflow pydantic pyyaml scikit-learn


In [ ]:
# =============================================================================
# CELL 3: EXTRACT PROJECT CODE
# =============================================================================
import zipfile
from pathlib import Path

# If project_for_colab.zip was uploaded, extract it
if os.path.exists('project_for_colab.zip'):
    print("Extracting uploaded project_for_colab.zip...")
    with zipfile.ZipFile('project_for_colab.zip', 'r') as zf:
        zf.extractall('.')
    print("✓ Project files extracted.")
elif is_colab and not os.path.exists('shelf_monitor'):
    from google.colab import files
    print("Please upload project_for_colab.zip from your local computer:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            with zipfile.ZipFile(fn, 'r') as zf:
                zf.extractall('.')
            print(f"✓ Extracted {fn}")

sys.path.insert(0, os.path.abspath('.'))
print("Project import path verified.")


In [ ]:
# =============================================================================
# CELL 4: PERSISTENCE & CHECKPOINTS
# =============================================================================
checkpoint_dir = Path('/kaggle/working/checkpoints') if is_kaggle else Path('checkpoints')
if is_colab:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        checkpoint_dir = Path('/content/drive/MyDrive/shelf_checkpoints')
        print(f"Mounted Google Drive. Checkpoints will be saved to: {checkpoint_dir}")
    except Exception as e:
        print(f"Drive mount skipped: {e}. Saving checkpoints locally.")

checkpoint_dir.mkdir(parents=True, exist_ok=True)
output_models_dir = Path('models_export')
output_models_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# =============================================================================
# CELL 5: DATA PREPARATION (SKU-110K & CUSTOM DATA)
# =============================================================================
from ultralytics import YOLO
from training.prepare_roboflow import prepare_roboflow_dataset

# Ultralytics automatically downloads and formats SKU-110K when passed 'SKU-110K.yaml'
sku110k_yaml = 'SKU-110K.yaml'
print("SKU-110K dataset ready via Ultralytics auto-download.")

# Prepare optional custom dataset
custom_yaml = None
if CUSTOM_DATA.get('zip_path') or CUSTOM_DATA.get('roboflow_api_key'):
    custom_dir = prepare_roboflow_dataset(
        zip_path=CUSTOM_DATA.get('zip_path'),
        api_key=CUSTOM_DATA.get('roboflow_api_key'),
        workspace=CUSTOM_DATA.get('workspace'),
        project=CUSTOM_DATA.get('project'),
        version=CUSTOM_DATA.get('version', 1),
        output_dir='data/custom_yolo'
    )
    if (custom_dir / 'data.yaml').exists():
        custom_yaml = str(custom_dir / 'data.yaml')
        print(f"✓ Custom dataset ready: {custom_yaml}")


In [ ]:
# =============================================================================
# CELL 6: STAGE 1 — PRETRAIN ON SKU-110K
# =============================================================================
from training.train_detector import train_detector

print("Starting Stage 1 Pretraining on SKU-110K...")
stage1_pt = train_detector(
    stage1_data=sku110k_yaml if not SMOKE_TEST else None,
    stage2_data=None,
    epochs_stage1=EPOCHS_STAGE1,
    batch=BATCH,
    imgsz=IMGSZ,
    model_size=MODEL_SIZE,
    device='0' if has_gpu else 'cpu',
    seed=SEED,
    output_dir=output_models_dir,
    smoke_test=SMOKE_TEST,
)
print(f"✓ Stage 1 model ready: {stage1_pt}")


In [ ]:
# =============================================================================
# CELL 7: STAGE 2 — FINE-TUNE ON CUSTOM STORE DATA
# =============================================================================
final_detector_pt = stage1_pt
if custom_yaml and os.path.exists(custom_yaml):
    print(f"Fine-tuning on custom data: {custom_yaml}...")
    final_detector_pt = train_detector(
        stage1_data=None,
        stage2_data=custom_yaml,
        epochs_stage2=EPOCHS_STAGE2,
        batch=BATCH,
        imgsz=IMGSZ,
        model_size=str(stage1_pt),
        device='0' if has_gpu else 'cpu',
        seed=SEED,
        output_dir=output_models_dir,
        smoke_test=SMOKE_TEST,
    )
else:
    print("No custom data specified. Using Stage 1 (SKU-110K) weights as final detector.")


In [ ]:
# =============================================================================
# CELL 8: EVALUATION & METRICS
# =============================================================================
from training.evaluate import evaluate_system

eval_data = custom_yaml if custom_yaml else sku110k_yaml
eval_metrics = evaluate_system(
    model_path=final_detector_pt,
    dataset_yaml=eval_data,
    out_dir='outputs/reports'
)
print("Evaluation Summary:", eval_metrics)


In [ ]:
# =============================================================================
# CELL 9: TRAIN SKU EMBEDDING NETWORK
# =============================================================================
from training.train_embedding import train_embedding

crop_source = SKU_CROPS_DIR or 'data/crops'
embedding_pt = train_embedding(
    data_dir=crop_source,
    epochs=EMBED_EPOCHS,
    batch_size=32 if has_gpu else 4,
    embedding_dim=256,
    device='cuda' if has_gpu else 'cpu',
    output_dir=output_models_dir,
    smoke_test=SMOKE_TEST,
)
print(f"✓ Embedding network weights: {embedding_pt}")


In [ ]:
# =============================================================================
# CELL 10: BUILD FAISS INDEX & SKU LABELS
# =============================================================================
from training.build_sku_index import build_index_from_crops

crop_dir = SKU_CROPS_DIR or ('outputs/scratch_crops' if os.path.exists('outputs/scratch_crops') else 'data/crops')
num_vecs, num_classes = build_index_from_crops(
    crops_dir=crop_dir,
    output_dir=output_models_dir,
    embedding_model_path=embedding_pt,
    embedding_dim=256,
)
print(f"✓ Indexed {num_vecs} vectors across {num_classes} SKUs.")


In [ ]:
# =============================================================================
# CELL 11: EXPORT ONNX & VERIFY PARITY
# =============================================================================
from training.export_onnx import export_detector_onnx, export_embedding_onnx

detector_onnx = export_detector_onnx(
    pt_path=final_detector_pt,
    out_path=output_models_dir / 'product_detector.onnx',
    imgsz=IMGSZ,
)

embedding_onnx = export_embedding_onnx(
    pt_path=embedding_pt,
    out_path=output_models_dir / 'sku_embedding.onnx',
    embedding_dim=256,
)
print("✓ Both models exported to ONNX successfully.")


In [ ]:
# =============================================================================
# CELL 12: VALIDATE CONTRACT ARTIFACTS
# =============================================================================
required_files = [
    'product_detector.pt',
    'product_detector.onnx',
    'sku_embedding.pt',
    'sku_embedding.onnx',
    'sku_index.faiss',
    'sku_labels.json',
]

all_present = True
print("Checking generated artifacts against contract:")
for fn in required_files:
    fp = output_models_dir / fn
    if fp.exists():
        print(f"  [✓] {fn:<24} : {fp.stat().st_size // 1024} KB")
    else:
        print(f"  [✗] MISSING: {fn}")
        all_present = False

assert all_present, "Missing one or more required model contract files!"
print("✓ All contract files verified.")


In [ ]:
# =============================================================================
# CELL 13: WRITE MANIFEST & BUNDLE shelf_models.zip
# =============================================================================
import json, zipfile
from datetime import datetime, timezone
from shelf_monitor.frameworks.model_registry import ModelManifest

manifest = ModelManifest(
    schema_version='1.0',
    created_at=datetime.now(timezone.utc).isoformat(),
    synthetic=SMOKE_TEST,
    detector={
        'file': 'product_detector.pt',
        'onnx_file': 'product_detector.onnx',
        'imgsz': IMGSZ,
        'conf_default': 0.35,
        'iou_default': 0.45,
        'max_det': 300,
        'class_names': ['product']
    },
    embedding={
        'file': 'sku_embedding.pt',
        'onnx_file': 'sku_embedding.onnx',
        'input_size': 224,
        'dim': 256,
        'backbone': 'mobilenet_v3_large'
    },
    index={
        'file': 'sku_index.faiss',
        'labels_file': 'sku_labels.json',
        'metric': 'inner_product',
        'dim': 256,
        'num_vectors': num_vecs,
        'similarity_threshold': 0.6
    },
    metrics=eval_metrics,
    training={
        'epochs_stage1': EPOCHS_STAGE1,
        'epochs_stage2': EPOCHS_STAGE2 if custom_yaml else 0,
        'dataset_notes': 'SKU-110K Pretrained + Custom fine-tuned' if custom_yaml else 'SKU-110K Pretrained'
    }
)

manifest_path = output_models_dir / 'model_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    f.write(manifest.model_dump_json(indent=2))
print(f"✓ Created {manifest_path}")

# Bundle into single zip file: shelf_models.zip
zip_path = 'shelf_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in output_models_dir.iterdir():
        if f.is_file():
            zf.write(f, arcname=f.name)

print(f"\n{'='*60}")
print(f"✓ SUCCESSFULLY BUNDLED: {zip_path} ({os.path.getsize(zip_path) // (1024*1024)} MB)")
print(f"{'='*60}")


In [ ]:
# =============================================================================
# CELL 14: DOWNLOAD AND NEXT STEPS
# =============================================================================
if is_colab:
    from google.colab import files
    print("Downloading shelf_models.zip to your computer...")
    files.download('shelf_models.zip')
elif is_kaggle:
    print("On Kaggle: Navigate to the 'Output' tab on the right panel and download 'shelf_models.zip'.")

print("""
===============================================================================
NEXT STEPS:
1. Download 'shelf_models.zip'
2. Extract its contents directly into your local 'models/' directory:
   DL-PROJECT/models/
     ├── product_detector.pt
     ├── product_detector.onnx
     ├── sku_embedding.pt
     ├── sku_embedding.onnx
     ├── sku_index.faiss
     ├── sku_labels.json
     └── model_manifest.json
3. In your local terminal, run:
   python -m shelf_monitor check-models
   python -m shelf_monitor ui
===============================================================================
""")
